# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os
import time
from datetime import datetime
from urllib.parse import urlparse
from dotenv import load_dotenv
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage, RetrievalEvaluation
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, VectorStore
from lib.memory import LongTermMemory, MemorySearchResult, MemoryFragment
from lib.evaluation import AgentEvaluator, TestCase
from lib import dashboard_logs
from typing import List, Optional, Dict, Any


In [3]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

#CONSTANTS
load_dotenv('.env')

CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

CHROMADB_PATH = "chromadb"
COLLECTION_NAME = "udaplay"

# Vector DB
vector_store_manager = VectorStoreManager(
    openai_api_key=CHROMA_OPENAI_API_KEY,
    persist_directory=CHROMADB_PATH,
)
game_store: VectorStore = vector_store_manager.get_or_create_store(COLLECTION_NAME)
long_term_memory = LongTermMemory(db=vector_store_manager)

USER_ID = "default_user"


In [6]:
import ast
import json
import re

GAME_DOC_PATTERN = re.compile(
    r"^\[(?P<platform>.+?)\]\s+(?P<name>.+?)\s+\((?P<year>[^)]+)\)\s+-\s+(?P<description>.*)$"
)
CITATION_PATTERN = re.compile(
    r"^(?P<name>.+?)\s+\((?P<platform>.+),\s+(?P<year>[^)]+)\)$"
)


def _unknown_record() -> Dict[str, Any]:
    return {
        "citation": "Unknown Game (Unknown Platform, Unknown Year)",
        "name": "Unknown Game",
        "platform": "Unknown Platform",
        "year": "Unknown Year",
        "description": "",
    }



def _record_from_citation_text(citation_text: str) -> Dict[str, Any]:
    match = CITATION_PATTERN.match((citation_text or '').strip())
    if not match:
        return _unknown_record()

    parsed = match.groupdict()
    name = parsed.get("name", "Unknown Game")
    platform = parsed.get("platform", "Unknown Platform")
    year = parsed.get("year", "Unknown Year")
    return {
        "citation": f"{name} ({platform}, {year})",
        "name": name,
        "platform": platform,
        "year": year,
        "description": "",
    }



def _record_from_document_text(document_text: str) -> Dict[str, Any]:
    match = GAME_DOC_PATTERN.match((document_text or '').strip())
    if not match:
        citation_record = _record_from_citation_text(document_text)
        if citation_record["name"] != "Unknown Game":
            return citation_record
        return _unknown_record()

    parsed = match.groupdict()
    name = parsed.get("name", "Unknown Game")
    platform = parsed.get("platform", "Unknown Platform")
    year = parsed.get("year", "Unknown Year")
    description = parsed.get("description", "")
    return {
        "citation": f"{name} ({platform}, {year})",
        "name": name,
        "platform": platform,
        "year": year,
        "description": description,
    }



def _record_from_sources(metadata: Optional[Dict[str, Any]], document_text: str) -> Dict[str, Any]:
    metadata = metadata or {}
    fallback = _record_from_document_text(document_text)

    name = metadata.get("Name") or fallback["name"]
    platform = metadata.get("Platform") or fallback["platform"]
    year = metadata.get("YearOfRelease") or fallback["year"]
    description = metadata.get("Description") or fallback["description"]

    return {
        "citation": f"{name} ({platform}, {year})",
        "name": name,
        "platform": platform,
        "year": year,
        "description": description,
    }



def _decode_possible_string(value: Any, max_passes: int = 3) -> Any:
    current = value
    for _ in range(max_passes):
        if not isinstance(current, str):
            break

        text = current.strip()
        if not text:
            return []

        try:
            decoded = json.loads(text)
        except json.JSONDecodeError:
            try:
                decoded = ast.literal_eval(text)
            except (ValueError, SyntaxError):
                break

        if decoded == current:
            break
        current = decoded

    return current



def _normalize_retrieved_docs(retrieved_docs: Any) -> List[Dict[str, Any]]:
    decoded = _decode_possible_string(retrieved_docs)

    if isinstance(decoded, str):
        pieces = [piece.strip() for piece in decoded.split(';') if piece.strip()]
        decoded = pieces if pieces else [decoded]

    if isinstance(decoded, dict):
        decoded = [decoded]

    if not isinstance(decoded, list):
        return []

    normalized_docs = []
    for item in decoded:
        if isinstance(item, dict):
            name = item.get("name") or item.get("Name") or "Unknown Game"
            platform = item.get("platform") or item.get("Platform") or "Unknown Platform"
            year = item.get("year") or item.get("YearOfRelease") or "Unknown Year"
            description = item.get("description") or item.get("Description") or ""
            citation = item.get("citation") or f"{name} ({platform}, {year})"
            normalized_docs.append({
                "citation": citation,
                "name": name,
                "platform": platform,
                "year": year,
                "description": description,
            })
        elif isinstance(item, str):
            normalized_docs.append(_record_from_document_text(item))

    return normalized_docs



def _all_unknown_records(records: List[Dict[str, Any]]) -> bool:
    return bool(records) and all(
        record.get("name") == "Unknown Game"
        and record.get("platform") == "Unknown Platform"
        and str(record.get("year")) == "Unknown Year"
        for record in records
    )


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game



In [4]:
@tool
def retrieve_game(query:str, n_results:int):
    """
    Search the local vector database for game records.

    Args:
        query: A question about the game industry.
        n_results: Number of results to return.

    Returns a list of citation-friendly records, each containing:
        citation: Compact citation label for the game record
        name: Game name
        platform: Platform name
        year: Release year
        description: Additional details about the game
    """
    results = game_store.query(
        query_texts=[query],
        n_results=n_results
    )

    retrieved_docs = results.get("documents", [])
    retrieved_metadatas = results.get("metadatas", [])
    documents = retrieved_docs[0] if retrieved_docs else []
    metadatas = retrieved_metadatas[0] if retrieved_metadatas else []

    formatted_docs = []
    for index, document_text in enumerate(documents):
        metadata = metadatas[index] if index < len(metadatas) else None
        formatted_docs.append(_record_from_sources(metadata, document_text))

    return formatted_docs


#### Evaluate Retrieval Tool

In [6]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

In [5]:
@tool
def evaluate_retrieval(question:str, retrieved_docs:list) -> RetrievalEvaluation:
    """
    Evaluate whether the retrieved documents are sufficient to answer the user's question.
    Call this after retrieve_game to decide if you can answer from internal knowledge or need web search.

    Args:
        question: The user's original question.
        retrieved_docs: List of document strings returned by retrieve_game.

    Returns:
        useful: True if the documents are sufficient to answer; False if web search is needed.
        description: Explanation of why the documents are or are not sufficient.
    """
    normalized_docs = _normalize_retrieved_docs(retrieved_docs)

    if not normalized_docs:
        return RetrievalEvaluation(
            useful=False,
            description="No retrieved documents were available after normalization, so web search is required."
        )

    if _all_unknown_records(normalized_docs):
        return RetrievalEvaluation(
            useful=False,
            description="The retrieved documents could not be parsed into valid game records, so web search is required."
        )

    judge_llm = LLM(
        model="gpt-4o-mini",
        temperature=0.0,
        api_key=OPENAI_API_KEY
    )

    docs_text = "\n\n".join(
        (
            f"- Citation: {doc['citation']}\n"
            f"  Name: {doc['name']}\n"
            f"  Platform: {doc['platform']}\n"
            f"  Year: {doc['year']}\n"
            f"  Description: {doc['description']}"
        )
        for doc in normalized_docs
    )

    system_msg = SystemMessage(
        content=(
            "You are a strict evaluator for a retrieval-augmented generation (RAG) system. "
            "Given a question and a set of retrieved documents, decide only whether those documents "
            "are sufficient to answer from the local dataset. "
            "Treat a matching game record with the correct release year as sufficient for questions like "
            "'when was this game released?' even if the local dataset does not include a month or exact day. "
            "Treat clearly matching records as sufficient for identification questions like 'which Mario game was first 3D?' "
            "Return the structured schema only."
        )
    )

    user_msg = UserMessage(
        content=(
            f"Question:\n{question}\n\n"
            f"Retrieved documents:\n{docs_text}\n\n"
            "Decide whether the local documents are enough to answer correctly without web search."
        )
    )

    ai_message = judge_llm.invoke(
        input=[system_msg, user_msg],
        response_format=RetrievalEvaluation
    )

    if isinstance(ai_message.content, RetrievalEvaluation):
        return ai_message.content

    return RetrievalEvaluation.model_validate_json(ai_message.content)


#### Game Web Search Tool

In [8]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 



In [7]:
@tool
def game_web_search(question:str) -> Dict:
    """
    Search the web for information about video games and return citation-ready results.
    Use only when evaluate_retrieval says local documents are insufficient.

    Args:
        question: The search query.

    Returns:
        answer: Tavily's direct answer if available
        citations: list of source titles/domains/URLs for final-answer citations
        results: simplified result list with title, url, domain, and content
    """
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

    search_result = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    raw_results = search_result.get("results", [])
    citations = []
    simplified_results = []

    for result in raw_results[:5]:
        url = result.get("url", "")
        domain = urlparse(url).netloc.replace("www.", "") if url else ""
        title = result.get("title", "")
        content = result.get("content", "")
        citations.append({
            "title": title,
            "domain": domain,
            "url": url,
        })
        simplified_results.append({
            "title": title,
            "domain": domain,
            "url": url,
            "content": content,
        })

    formatted_results = {
        "answer": search_result.get("answer", ""),
        "citations": citations,
        "results": simplified_results,
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }

    return formatted_results


#### Memory Store And Retrieval Tool 

In [8]:
@tool
def retrieve_memory(question: str, limit: int = 3) -> Dict:
    """
    Retrieve relevant long-term memories for the current user.

    args:
    - question: the user's current question.
    - limit: max number of memory fragments to return.

    Returns:
    - fragments: list of strings (memory contents)
    - metadata: distances or any other scores
    """
    result = long_term_memory.search(
        query_text=question,
        owner=USER_ID,
        limit=limit,
        namespace="default",
    )

    return {
        "fragments": [f.content for f in result.fragments],
        "metadata": result.metadata,
    }

In [9]:
@tool
def store_memory(content: str, namespace: str = "default") -> Dict:
    """
    Store a new long-term memory for the current user.

    args:
    - content: the text to remember.
    - namespace: optional logical grouping.

    Returns:
    - success: bool
    """
    fragment = MemoryFragment(
        content=content,
        owner=USER_ID,
        namespace=namespace,
    )
    long_term_memory.register(fragment)
    return {"success": True}

### Agent

In [10]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

tools = [
    retrieve_game, 
    evaluate_retrieval, 
    game_web_search, 
    retrieve_memory, 
    store_memory
    ]

# instructions = (
# "You are a research agent for the video game industry."
# "You can answer multistep questions by sequentially calling functions for:"
# "- Answer questions using internal knowledge (RAG)"
# "- call retrieve_memory at the beginning when prior context may matter"
# "- Run the evaluate_retrieval tool to see if the retrieved information is enought to answer"
# "- Search the web when the internal knowledge is insufficient using the game web search"
# "- After answering, if it discovers a user preference or durable fact, call store_memory with a short text summary."
# "You follow a pattern of of Thought and Action. "
# "Create a plan of execution: "
# "- Use Thought to describe your thoughts about the question you have been asked. "
# "- Use Action to specify one of the tools available to you. if you don't have a tool available, you can respond directly."
# "When you think it's over, return the answer "
# "Never try to respond directly if the question needs a tool. "
# "But if you don't have a tool available, you can respond directly. "
# f"The actions you have are the Tools: {tools}. \n"
# )

# I used GPT5.4 to help me imporve the prompt based on the initial prompt
# 

improved_instructions = f"""
You are UdaPlay, a research agent for the video game industry.

You must answer factual questions by following the required workflow below.

REQUIRED WORKFLOW FOR FACTUAL QUESTIONS
1. First call `retrieve_game`.
2. Second call `evaluate_retrieval` using the original user question and the documents returned by `retrieve_game`. Pass the full JSON list returned by `retrieve_game` as `retrieved_docs` instead of summarizing it.
3. If `evaluate_retrieval.useful` is True, answer from internal retrieval and do not call `game_web_search`.
4. If `evaluate_retrieval.useful` is False, call `game_web_search` and answer using the best available evidence.

HARD RULES
- Do not skip `retrieve_game` for factual questions.
- Do not call `game_web_search` before `evaluate_retrieval`.
- Use `retrieve_memory` only when the question depends on prior user-specific context.
- Use `store_memory` only for durable user-specific preferences or personal facts.
- Never store public game facts in memory.

ANSWER FORMAT
- Start with a direct answer to the question.
- Optionally add one short explanatory paragraph if it helps.
- End with a final line that begins exactly with `Sources:`.
- When citing internal retrieval, use the `citation` field from `retrieve_game`. Example: `Sources: Internal game data — Super Mario 64 (Nintendo 64, 1996)`.
- When citing web evidence, cite 1-2 named sources from `game_web_search["citations"]` using source titles or domains. Example: `Sources: PlayStation Store; Wikipedia`.
- If both internal retrieval and web search were used, include both in the same `Sources:` line separated by semicolons.
- Never write only `verified with web search`; always name the sources.

QUALITY BAR
- Be concise, factual, and helpful.
- If evidence is incomplete or ambiguous, say so clearly instead of guessing.
- Prefer the shortest valid workflow that still produces a correct, cited answer.

Available tools: {tools}
"""


In [11]:

agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions=improved_instructions,
)

In [13]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

q1 = "When Pokémon Gold and Silver was released?"
q2 = "Which one was the first 3D platformer Mario game?"
q3 = "Was Mortal Kombat X realeased for Playstation 5?"
q4 = "God of War is my all time favorite game. What are the 2026 onward releases expected for the God of War franchise?"
q5 = "I liked the first season of the live series the Last of Us, why the second went so bad? are the games better than the live serie?"


### Retrieval Diagnostics

Run this cell after setup to inspect what Chroma is returning before the agent or evaluator transform the results. The local-only benchmark queries should surface `Pokémon Gold and Silver` and `Super Mario 64` near the top.

In [ ]:
diagnostic_queries = [
    ("gold_silver_release", q1),
    ("first_3d_mario", q2),
    ("mkx_ps5", q3),
    ("edge_case", "Was The Elder Scrolls V: Skyrim released for Nintendo Switch?"),
]

for case_id, query in diagnostic_queries:
    print(f"\n=== {case_id}: {query} ===")
    results = game_store.query(query_texts=[query], n_results=3)
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    if not documents:
        print("No documents returned.")
        continue

    for index, document_text in enumerate(documents):
        metadata = metadatas[index] if index < len(metadatas) else None
        distance = distances[index] if index < len(distances) else None
        print(f"Result {index + 1} | distance={distance}")
        print(f"document: {document_text}")
        print(f"metadata: {metadata}")
        print("-" * 80)


In [15]:
demo_queries = [
    ("demo_1", q1),
    ("demo_1", q2),
    ("demo_2", q3),
    ("demo_3", q4),
    ("demo_3", q5),
]

interactive_results = []

for session_id, query in demo_queries:
    start_time = time.perf_counter()
    run_object = agent.invoke(query=query, session_id=session_id)
    execution_time = time.perf_counter() - start_time

    dashboard_logs.log_agent_run_from_run(
        run_object,
        session_id=session_id,
        query=query,
        execution_time_sec=execution_time,
        source="interactive",
    )

    interactive_results.append({
        "session_id": session_id,
        "query": query,
        "messages": run_object.get_final_state()["messages"],
    })

for result in interactive_results:
    print(f"Session: {result['session_id']}")
    print(f"Query: {result['query']}")
    print(result["messages"])
    print("-" * 80)

interactive_results


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachi

[{'session_id': 'demo_1',
  'query': 'When Pokémon Gold and Silver was released?',
  'messages': [SystemMessage(role='system', content='\nYou are UdaPlay, a research agent for the video game industry.\n\nYou must answer factual questions by following the required workflow below.\n\nREQUIRED WORKFLOW FOR FACTUAL QUESTIONS\n1. First call `retrieve_game`.\n2. Second call `evaluate_retrieval` using the original user question and the documents returned by `retrieve_game`. Pass the full JSON list returned by `retrieve_game` as `retrieved_docs` instead of summarizing it.\n3. If `evaluate_retrieval.useful` is True, answer from internal retrieval and do not call `game_web_search`.\n4. If `evaluate_retrieval.useful` is False, call `game_web_search` and answer using the best available evidence.\n\nHARD RULES\n- Do not skip `retrieve_game` for factual questions.\n- Do not call `game_web_search` before `evaluate_retrieval`.\n- Use `retrieve_memory` only when the question depends on prior user-speci

### Evaluation

This section benchmarks the agent on a fixed set of test cases using three complementary views:
- final-response evaluation,
- single-step tool evaluation,
- and full trajectory evaluation.

Each benchmark run uses an isolated `session_id` and `USER_ID` so prior memory does not leak across cases.

In [16]:
evaluator = AgentEvaluator()

benchmark_cases = [
    TestCase(
        id="gold_silver_release",
        description="Check whether the agent answers a local release-date question using retrieval.",
        user_query=q1,
        expected_tools=["retrieve_game", "evaluate_retrieval"],
        reference_answer="Pokemon Gold and Silver were first released in Japan in 1999.",
        max_steps=5,
    ),
    TestCase(
        id="first_3d_mario",
        description="Check whether the agent identifies the first 3D Mario platformer from the local dataset.",
        user_query=q2,
        expected_tools=["retrieve_game", "evaluate_retrieval"],
        reference_answer="Super Mario 64 was the first 3D Mario platformer.",
        max_steps=5,
    ),
    TestCase(
        id="mkx_ps5",
        description="Check whether the agent falls back to web search for a platform/version question that is not well covered locally.",
        user_query=q3,
        expected_tools=["retrieve_game", "evaluate_retrieval", "game_web_search"],
        reference_answer="Mortal Kombat X was not released for PlayStation 5.",
        max_steps=6,
    ),
    TestCase(
        id="ambiguous_case",
        description="Check how the agent handles a broad and ambiguous release question.",
        user_query="When was Pokemon released?",
        expected_tools=["retrieve_game", "evaluate_retrieval"],
        reference_answer=None,
        max_steps=5,
    ),
    TestCase(
        id="edge_case",
        description="Check whether the agent uses web fallback for a stable platform/version question outside the local dataset.",
        user_query="Was The Elder Scrolls V: Skyrim released for Nintendo Switch?",
        expected_tools=["retrieve_game", "evaluate_retrieval", "game_web_search"],
        reference_answer="Yes. The Elder Scrolls V: Skyrim was released for Nintendo Switch.",
        max_steps=6,
    ),
]

In [17]:
def get_final_response_text(run):
    final_state = run.get_final_state() or {}
    messages = final_state.get("messages", [])

    for message in reversed(messages):
        if isinstance(message, AIMessage) and message.content:
            return message.content

    return ""


def get_tool_names(messages):
    tool_names = []

    for message in messages:
        if isinstance(message, AIMessage) and message.tool_calls:
            tool_names.extend(tool_call.function.name for tool_call in message.tool_calls)

    return tool_names


def run_benchmark_case(test_case):
    global USER_ID

    original_user_id = USER_ID
    USER_ID = f"eval_{test_case.id}"
    session_id = f"eval_{test_case.id}"

    if session_id in agent.memory.get_all_sessions():
        agent.reset_session(session_id)
    else:
        agent.memory.create_session(session_id)

    try:
        start_time = time.perf_counter()
        run = agent.invoke(query=test_case.user_query, session_id=session_id)
        execution_time = time.perf_counter() - start_time

        final_state = run.get_final_state() or {}
        messages = final_state.get("messages", [])
        final_response = get_final_response_text(run)
        tool_names = get_tool_names(messages)
        total_tokens = final_state.get("total_tokens", 0)

        final_eval = evaluator.evaluate_final_response(
            test_case=test_case,
            agent_response=final_response,
            execution_time=execution_time,
            total_tokens=total_tokens,
        )
        step_eval = evaluator.evaluate_single_step(messages, test_case.expected_tools)
        trajectory_eval = evaluator.evaluate_trajectory(test_case, run)

        return {
            "test_case": test_case,
            "run": run,
            "final_response": final_response,
            "tool_names": tool_names,
            "final_eval": final_eval,
            "step_eval": step_eval,
            "trajectory_eval": trajectory_eval,
            "execution_time": execution_time,
            "total_tokens": total_tokens,
        }
    finally:
        USER_ID = original_user_id


In [18]:
evaluation_results = []
benchmark_run_id = dashboard_logs.create_benchmark_run_id()

for case in benchmark_cases:
    result = run_benchmark_case(case)
    evaluation_results.append(result)
    session_id = f"eval_{result['test_case'].id}"
    ts = datetime.utcnow().isoformat() + "Z"
    dashboard_logs.log_agent_run_from_run(
        result["run"],
        session_id=session_id,
        query=result["test_case"].user_query,
        execution_time_sec=result["execution_time"],
        source="eval",
        benchmark_run_id=benchmark_run_id,
        case_id=result["test_case"].id,
    )
    dashboard_logs.log_eval_case(
        benchmark_run_id=benchmark_run_id,
        case_id=result["test_case"].id,
        run_id=result["run"].run_id,
        timestamp=ts,
        source="eval",
        query=result["test_case"].user_query,
        tools_used=result["tool_names"],
        final_score=result["final_eval"].overall_score,
        trajectory_score=result["trajectory_eval"].overall_score,
        step_tool_correct=result["step_eval"].tool_interaction.correct_tool_selected,
        task_completed=result["trajectory_eval"].task_completion.task_completed,
        execution_time=result["execution_time"],
        total_tokens=result["total_tokens"],
        feedback=result["trajectory_eval"].feedback,
    )

summary_rows = []
for result in evaluation_results:
    row = {
        "case_id": result["test_case"].id,
        "query": result["test_case"].user_query,
        "tools_used": result["tool_names"],
        "final_score": result["final_eval"].overall_score,
        "trajectory_score": result["trajectory_eval"].overall_score,
        "step_tool_correct": result["step_eval"].tool_interaction.correct_tool_selected,
        "task_completed": result["trajectory_eval"].task_completion.task_completed,
        "execution_time": round(result["execution_time"], 3),
        "total_tokens": result["total_tokens"],
        "feedback": result["trajectory_eval"].feedback,
    }
    summary_rows.append(row)

ts_summary = datetime.utcnow().isoformat() + "Z"
dashboard_logs.log_eval_summary(
    benchmark_run_id=benchmark_run_id,
    timestamp=ts_summary,
    total_cases=len(summary_rows),
    mean_final_score=sum(r["final_score"] for r in summary_rows) / len(summary_rows) if summary_rows else None,
    mean_trajectory_score=sum(r["trajectory_score"] for r in summary_rows) / len(summary_rows) if summary_rows else None,
    task_completed_count=sum(1 for r in summary_rows if r["task_completed"]),
    total_tokens=sum(r["total_tokens"] for r in summary_rows),
)

for row in summary_rows:
    print(f"Case: {row['case_id']}")
    print(f"  Query: {row['query']}")
    print(f"  Tools used: {row['tools_used']}")
    print(f"  Final score: {row['final_score']:.2f}")
    print(f"  Trajectory score: {row['trajectory_score']:.2f}")
    print(f"  Step tool correct: {row['step_tool_correct']}")
    print(f"  Task completed: {row['task_completed']}")
    print(f"  Execution time: {row['execution_time']}s")
    print(f"  Total tokens: {row['total_tokens']}")
    print(f"  Feedback: {row['feedback']}")
    print('-' * 80)

summary_rows

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


/var/folders/04/chwb3n9s59j8f4451k4gvy800000gn/T/ipykernel_57072/756804380.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat() + "Z"


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor


/var/folders/04/chwb3n9s59j8f4451k4gvy800000gn/T/ipykernel_57072/756804380.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts_summary = datetime.utcnow().isoformat() + "Z"


[{'case_id': 'gold_silver_release',
  'query': 'When Pokémon Gold and Silver was released?',
  'tools_used': ['retrieve_game', 'evaluate_retrieval', 'game_web_search'],
  'final_score': 1.0,
  'trajectory_score': 0.75,
  'step_tool_correct': False,
  'task_completed': False,
  'execution_time': 19.412,
  'total_tokens': 9445,
  'feedback': "Trajectory: 8 steps, Tools used: ['retrieve_game', 'evaluate_retrieval', 'game_web_search'], Expected: ['retrieve_game', 'evaluate_retrieval']"},
 {'case_id': 'first_3d_mario',
  'query': 'Which one was the first 3D platformer Mario game?',
  'tools_used': ['retrieve_game', 'evaluate_retrieval'],
  'final_score': 1.0,
  'trajectory_score': 0.75,
  'step_tool_correct': True,
  'task_completed': False,
  'execution_time': 21.888,
  'total_tokens': 4565,
  'feedback': "Trajectory: 6 steps, Tools used: ['retrieve_game', 'evaluate_retrieval'], Expected: ['retrieve_game', 'evaluate_retrieval']"},
 {'case_id': 'mkx_ps5',
  'query': 'Was Mortal Kombat X rea

In [19]:
failing_results = [
    result
    for result in evaluation_results
    if (
        result["final_eval"].overall_score < 0.75
        or result["trajectory_eval"].overall_score < 0.75
        or result["step_eval"].tool_interaction.correct_tool_selected is False
    )
]

if not failing_results:
    print("No weak runs found with the current benchmark threshold.")
else:
    for result in failing_results:
        test_case = result["test_case"]
        print(f"Case: {test_case.id}")
        print(f"  Expected tools: {test_case.expected_tools}")
        print(f"  Actual tools: {result['tool_names']}")
        print(f"  Final response: {result['final_response']}")
        print(f"  Final evaluation feedback: {result['final_eval'].feedback}")
        print(f"  Trajectory evaluation feedback: {result['trajectory_eval'].feedback}")
        print('-' * 80)

Case: gold_silver_release
  Expected tools: ['retrieve_game', 'evaluate_retrieval']
  Actual tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
  Final response: Pokémon Gold and Silver were released in Japan on November 21, 1999, followed by their North American release on October 15, 2000. These games introduced 100 new Pokémon and were compatible with both the Game Boy and Game Boy Color.

Sources: Wikipedia; Bulbapedia; IGN.
  Final evaluation feedback: The agent successfully answered the user's query about the release date of Pokémon Gold and Silver, providing specific dates for both Japan and North America. The format is correct, including a clear 'Sources' line that cites specific references. Additionally, the response follows the implicit instruction to use named citations rather than vague phrases.
  Trajectory evaluation feedback: Trajectory: 8 steps, Tools used: ['retrieve_game', 'evaluate_retrieval', 'game_web_search'], Expected: ['retrieve_game', 'evaluate_r

### (Optional) Advanced

In [20]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes